# ProcessBehavior Plotting Demo

This notebook demonstrates the powerful plotting capabilities of the ProcessBehavior library.

## Features Covered

1. **Built-in Themes** - Professional, ggplot, minimal, dark, publication
2. **Zone Shading** - Visual zones at ±1σ, ±2σ, ±3σ
3. **Two-Tier Signal Highlighting** - Red for limit violations, orange for pattern signals
4. **Run Rules Visualization** - Western Electric Rules 2-8
5. **Statistics Box** - Display n, CL, UCL, LCL
6. **Aspect Ratio Control** - Consistent chart proportions
7. **Residual Diagnostics** - Histogram, Q-Q, sequence plots
8. **Effects Visualization** - Factor and time main effects
9. **Report Generation** - Comprehensive HTML reports
10. **Custom Themes** - Full control over appearance

In [ ]:
import numpy as np
import pandas as pd

from processbehavior import ProcessDataFrame, ChartTheme, get_theme, list_themes
from processbehavior.plotting import Plotter

## Create Sample Data

We'll create two datasets:
1. Simple I-MR data (individual measurements)
2. Subgrouped data (replicated measurements within subgroups)

In [ ]:
# Set seed for reproducibility
np.random.seed(42)

# I-MR data with signals that trigger BOTH tiers:
# - Rule 1 (Red): Points outside control limits
# - Rules 2-8 (Orange): Pattern-based signals
imr_data = pd.DataFrame({
    'measurement': [
        # Baseline around 100 (points 0-19)
        98, 101, 99, 102, 100, 97, 103, 99, 101, 98,
        102, 100, 99, 101, 98, 103, 100, 99, 102, 101,
        # Run above center - triggers Rule 4 (points 20-29)
        104, 105, 104, 106, 105, 104, 105, 106, 104, 105,
        # Back to normal (points 30-39)
        99, 101, 100, 99, 102, 98, 101, 100, 99, 102,
        # Outside limits - triggers Rule 1 (points 40-42)
        115, 85, 116
    ],
    'time': range(43)
})

print("I-MR Data Shape:", imr_data.shape)
print("This data triggers both signal tiers:")
print("  - Rule 1 (Red): Points 40-42 outside limits")
print("  - Rule 4 (Orange): Points 20-29 run above center")
imr_data.head()

In [ ]:
# Subgrouped data with replication
n_subgroups = 25
subgroup_size = 5

xbar_data = pd.DataFrame({
    'measurement': np.random.normal(50, 3, n_subgroups * subgroup_size),
    'subgroup': np.repeat(range(n_subgroups), subgroup_size),
    'operator': np.tile(['Alice', 'Bob'], n_subgroups * subgroup_size // 2 + 1)[:n_subgroups * subgroup_size]
})

print("Subgrouped Data Shape:", xbar_data.shape)
xbar_data.head()

## 1. Built-in Themes

ProcessBehavior includes 5 professionally designed themes inspired by ggplot2's philosophy.

In [ ]:
# List available themes
print("Available themes:", list_themes())

In [ ]:
# Run analysis using formulate() -> analyze()
pdf = ProcessDataFrame(imr_data)
study = pdf.formulate(response=pdf.columns.measurement, time=pdf.columns.time)
result = study.analyze()

print(f"SDS: {study.sds}, Recommended: {study.recommended_chart}")
print(result)

In [ ]:
# Default theme: processbehavior
fig = result.plot(title='Default Theme (processbehavior)')
fig.show()

In [ ]:
# ggplot theme - inspired by R's ggplot2
fig = result.plot(theme='ggplot', title='ggplot Theme')
fig.show()

In [ ]:
# Minimal theme - clean, reduced visual elements
fig = result.plot(theme='minimal', title='Minimal Theme')
fig.show()

In [ ]:
# Dark theme - for presentations and dark mode UIs
fig = result.plot(theme='dark', title='Dark Theme')
fig.show()

In [ ]:
# Publication theme - optimized for academic papers and print
fig = result.plot(theme='publication', title='Publication Theme')
fig.show()

## 2. Zone Shading

Visualize the zones used in Western Electric rules:
- **Zone C** (green): Center ±1σ - normal variation
- **Zone B** (yellow): ±1σ to ±2σ - watch
- **Zone A** (red): ±2σ to ±3σ - warning

In [ ]:
# Enable zone shading
fig = result.plot(
    show_zones=True,
    title='Control Chart with Zone Shading'
)
fig.show()

In [ ]:
# Zone shading with dark theme
fig = result.plot(
    show_zones=True,
    theme='dark',
    title='Zone Shading with Dark Theme'
)
fig.show()

## 3. Two-Tier Signal Highlighting

ProcessBehavior uses a **two-tier color system** for signal visualization. Both signal types use the same circle marker style as regular data points, differentiated only by color for a clean, professional appearance.

| Tier | Rules | Color | Meaning |
|------|-------|-------|---------|
| **Hard Signal** | Rule 1 | Red ● | Point outside control limits (3σ) |
| **Pattern Signal** | Rules 2-8 | Orange ● | Pattern-based signals |

**Western Electric Rules detected:**
- Rule 1: Point beyond control limits
- Rule 2: 2 of 3 points in Zone A
- Rule 3: 4 of 5 points in Zone B or beyond
- Rule 4: 8 consecutive points on same side of center
- Rule 5: 6 points in a row trending up or down
- And more...

**In our demo data:**
- Points 40-42 trigger **Rule 1** (red) - outside control limits
- Points 20-29 trigger **Rule 4** (orange) - 8+ consecutive points above center line

In [ ]:
# Show run rules (Western Electric Rules 2-8)
fig = result.plot(
    show_rules=True,
    title='Control Chart with Run Rules'
)
fig.show()

In [ ]:
# Combine zones and rules for complete visualization
fig = result.plot(
    show_zones=True,
    show_rules=True,
    title='Complete Control Chart: Zones + Rules'
)
fig.show()

## 4. Statistics Box

Display key statistics (n, CL, UCL, LCL) in a clean box overlay.

In [ ]:
# Enable stats box
fig = result.plot(
    show_stats=True,
    title='Control Chart with Statistics Box'
)
fig.show()

In [ ]:
# Full featured chart
fig = result.plot(
    show_zones=True,
    show_rules=True,
    show_stats=True,
    title='Full Featured Control Chart'
)
fig.show()

## 5. Aspect Ratio Control

Control chart proportions for consistent presentation.

In [ ]:
# 16:9 widescreen aspect ratio
fig = result.plot(
    width=1200,
    aspect_ratio=16/9,
    title='16:9 Widescreen Format'
)
fig.show()

In [ ]:
# Square format
fig = result.plot(
    width=600,
    aspect_ratio=1.0,
    title='Square Format (1:1)'
)
fig.show()

## 6. Subgrouped Data Analysis

In [ ]:
# Analyze subgrouped data
pdf_xbar = ProcessDataFrame(xbar_data)
study_xbar = pdf_xbar.formulate(
    response=pdf_xbar.columns.measurement,
    factors=[pdf_xbar.columns.subgroup]
)
result_xbar = study_xbar.analyze()

print(f"SDS: {study_xbar.sds}, Recommended: {study_xbar.recommended_chart}")
print(result_xbar)

In [ ]:
# Plot control charts for subgrouped data
fig = result_xbar.plot(
    show_zones=True,
    show_stats=True
)
fig.show()

## 7. Residual Diagnostics

Visualize VAS residuals to assess process behavior and identify patterns.

In [ ]:
# Check if residuals are available
print(f"Has residuals: {result_xbar.has_residuals}")

if result_xbar.has_residuals:
    print(f"Available residuals: {list(result_xbar.residuals.columns)}")

In [ ]:
# Residual diagnostic plots (all three)
if result_xbar.has_residuals:
    plotter = Plotter(result_xbar)
    fig = plotter.plot_residuals(residual_type='R1', plot_type='all')
    fig.show()

In [ ]:
# Individual residual plots
if result_xbar.has_residuals:
    # Histogram only
    fig = plotter.plot_residuals(plot_type='histogram')
    fig.show()

In [ ]:
# Q-Q plot for normality check
if result_xbar.has_residuals:
    fig = plotter.plot_residuals(plot_type='qq')
    fig.show()

## 8. Effects Visualization

Visualize factor and time main effects to identify sources of variation.

In [ ]:
# Check if effects are available
print(f"Has effects: {result_xbar.has_effects}")

if result_xbar.has_effects:
    print(f"Available effects: {list(result_xbar.effects.keys())}")

In [ ]:
# Factor effects bar chart
if result_xbar.has_effects:
    fig = plotter.plot_effects(effect_type='factor')
    fig.show()

## 9. Custom Themes

Create your own themes for complete control over appearance.

In [ ]:
# Inspect a theme's properties
theme = get_theme('processbehavior')
print(f"Theme: {theme.name}")
print(f"Data color: {theme.data_color}")
print(f"Data marker size: {theme.data_marker_size}")
print(f"Signal color (Rule 1): {theme.signal_color}")
print(f"Signal marker symbol: {theme.signal_marker_symbol}")
print(f"Pattern signal color (Rules 2-8): {getattr(theme, 'pattern_signal_color', 'N/A')}")
print(f"Center color: {theme.center_color}")
print(f"Zone opacity: {theme.zone_opacity}")

In [ ]:
# Create a custom corporate theme
corporate_theme = ChartTheme(
    name='corporate',
    # Data appearance
    data_color='#003366',       # Navy blue
    data_marker_size=8,
    data_line_width=2.5,
    # Control limits
    ucl_color='#CC0000',        # Corporate red
    lcl_color='#CC0000',
    center_color='#006633',     # Corporate green
    # Signals - same marker style as data, just different fill color
    signal_color='#CC0000',           # Red for Rule 1 (limit violations)
    signal_marker_size=8,             # Same size as data points
    signal_marker_symbol='circle',    # Same style as data points
    # Zones
    zone_a_color='#FFCCCC',
    zone_b_color='#FFFFCC',
    zone_c_color='#CCFFCC',
    zone_opacity=0.25,
    # Layout
    plot_bgcolor='#F8F8F8',
    paper_bgcolor='white',
    # Typography
    font_family='Georgia, serif',
    title_font_size=18,
)

# Use custom theme with rules to show two-tier signals
fig = result.plot(
    theme=corporate_theme,
    show_zones=True,
    show_rules=True,
    show_stats=True,
    title='Custom Corporate Theme with Two-Tier Signals'
)
fig.show()

In [ ]:
# Register custom theme for reuse
from processbehavior import register_theme

register_theme(corporate_theme)

# Now you can use it by name
fig = result.plot(theme='corporate', title='Using Registered Theme')
fig.show()

## 10. Report Generation

Generate comprehensive HTML reports with all visualizations.

In [ ]:
# Generate a full analysis report
plotter = Plotter(result)
plotter.generate_report(
    'analysis_report.html',
    title='Process Behavior Analysis Report',
    theme='processbehavior'
)

print("Report generated: analysis_report.html")
print("Open this file in a web browser to view the interactive report.")

In [ ]:
# Generate report with dark theme
plotter.generate_report(
    'dark_report.html',
    title='Dark Theme Report',
    theme='dark'
)

print("Dark report generated: dark_report.html")

## Summary

The ProcessBehavior plotting system provides:

| Feature | Description |
|---------|-------------|
| **5 Built-in Themes** | processbehavior, ggplot, minimal, dark, publication |
| **Zone Shading** | Visual ±1σ, ±2σ, ±3σ zones |
| **Two-Tier Signals** | Red ● for limit violations (Rule 1), Orange ● for patterns (Rules 2-8) |
| **Run Rules** | Western Electric Rules 2-8 visualization |
| **Stats Box** | n, CL, UCL, LCL display |
| **Aspect Ratio** | Consistent proportions (16:9, 4:3, etc.) |
| **Residual Plots** | Histogram, Q-Q, sequence diagnostics |
| **Effects Plots** | Factor and time main effects |
| **Report Generation** | Comprehensive HTML reports |
| **Custom Themes** | Full control via ChartTheme dataclass |

**Signal Marker Design:** All signals use the same circle marker style as regular data points, differentiated only by color. This provides a clean, professional appearance while making signals easy to identify.

All charts are interactive (zoom, pan, hover) and export to HTML, PNG, SVG, and PDF.

In [ ]:
# Save as interactive HTML
fig = result.plot(
    show_zones=True,
    show_stats=True,
    title='Exportable Chart'
)

# Save HTML (interactive)
fig.save_html('control_chart.html')
print("Saved: control_chart.html")

In [ ]:
# Save as static image (requires kaleido)
try:
    fig.figure.write_image('control_chart.png', width=1200, height=600)
    print("Saved: control_chart.png")
except Exception as e:
    print(f"Image export requires kaleido: pip install kaleido")